# A2 Kaggle v3 — Chỉ chạy hai seed còn lại: 67 và 69

**Seed69420 đã hoàn tất U/S0/S1. Không chạy lại pilot, main hoặc inference của seed đó.**
File này chỉ chạy **U-Net, S0, S1 × seeds67/69 = 6 run mới**; các điểm seed69420 đã xác minh được nhúng sẵn dưới dạng JSON, không cần upload checkpoint cũ.

1. Import notebook; bật **GPU + Internet**. Giữ GPU type và environment image qua các phiên.
2. Chạy **Save Version → Save & Run All**. Run all tương tác cũng được nhưng cần lưu/tải output trước khi mất phiên.
3. Mặc định dừng sau khoảng8 giờ tại epoch đã lưu rồi xuất ZIP. Có thể cần nhiều phiên; đây là ngân sách phần mềm, không phải cam kết về quota Kaggle.
4. Phiên sau Add Input output mới nhất của **v3**, giữ suite tag và chạy lại để resume. V3 có tag riêng cho kế hoạch6 run; không tự ghi đè/đổi ledger của v1/v2.

Giữ bản sửa ensurepip của v2: venv --without-pip + external pip --python/fallback zipapp PyPA.
Batch4/max40/patience8,512canvas,FP32,split36, AdamW U lr.001/S0,S1 lr6e-5,wd.01,CE+Dice1+1 không đổi.
S0/S1 cùng seed/init trong từng cặp và chỉ khác augmentation màu train. Pilot không dùng làm main weights.

**Tổng hợp:** bảng Kaggle riêng có2 seed; bảng kết hợp có seed69420 Colab +67/69 Kaggle khi có đủ kết quả.
Mean±SD kết hợp được gắn nhãn có khác biệt môi trường; không gọi đó là biến động chỉ do seed, và không gộp timing Colab/Kaggle.


In [ ]:
# 1. Settings — mặc định chỉ chạy 6 run còn lại. Giữ nguyên SUITE_TAG khi resume.
from pathlib import Path
import time
SESSION_STARTED_UNIX = time.time()
SUITE_TAG = 'A2_kaggle_remaining_67_69_v3'
MAX_SESSION_HOURS = 8.0   # dừng sau epoch đã lưu; có thể điều chỉnh theo quota/session của bạn
RUN_MAIN = True
RUN_EXTRAS = True        # prediction gallery + GPU forward benchmark sau test
SELECT_RUNS = None       # None = đủ 6; tùy chọn ['67/U', '67/S0', '67/S1', '69/U', '69/S0', '69/S1']
DATA_ARCHIVE = ''        # auto tìm /kaggle/input/**/dataset.zip, nếu không có thì tải Drive
RESTORE_FROM = ''        # auto tìm output suite đã gắn; hoặc đường dẫn suite/ZIP cụ thể
assert SUITE_TAG and all(c.isalnum() or c in '_-' for c in SUITE_TAG)
assert isinstance(MAX_SESSION_HOURS, (int, float)) and MAX_SESSION_HOURS > 0
assert Path('/kaggle/working').is_dir(), 'Notebook này dành cho Kaggle.'
OUTPUT_ROOT = Path('/kaggle/working/dlbench_a2_kaggle') / SUITE_TAG


## 2. Khôi phục code và môi trường

Notebook dùng một Python venv riêng kế thừa torch/torchvision GPU của Kaggle, cài các dependency đã pin trong venv.
Venv được tạo **không gọi ensurepip**. Pip bên ngoài quản lý interpreter này bằng `--python`;
nếu pip hệ thống thiếu/quá cũ, tải zipapp chính thức của PyPA vào thư mục tạm riêng.
Tham khảo: [pip --python cho venv chưa có pip](https://pip.pypa.io/en/stable/topics/python-option/).
Kernel không phải restart để đổi Pillow/Transformers. Torch/torchvision không được tự nâng cấp.
Code nhúng đối chiếu byte-for-byte với revision `7df3236`; metadata ghi rõ đây là snapshot nhúng, không phải Git checkout đang hoạt động.


In [ ]:
# Embedded verified source, helpers, split and completed69420 scores.
import base64, hashlib, json, os, sys, zlib
PAYLOAD = '...'
EXPECTED_SHA256 = {'src/dlbench/a2/__init__.py': '725c6ef4f9c6f219a46a259e99cffda45bfddb35682066713c1cdbb31d7a963e', 'src/dlbench/a2/cli.py': 'a24ab286afa6966cefc89b4e924bf47bdfa4c008dba915ae208460eaa7ca5fbc', 'src/dlbench/a2/comparison.py': '78461e39ff8dc4dae06c52fb3ce0354fe6e2a5c9ca6fd832d166ccd97c1d3b92', 'src/dlbench/a2/config.py': 'cdfedf36447a2b1bc61347e7d4ee75f53748770b7ecb19a79c6daf12b0bffe80', 'src/dlbench/a2/converter.py': 'a102c558cb97323803bd262baef61dca92e6572a8c8fa9b4eedb77a8576f0d50', 'src/dlbench/a2/data.py': '8e38f3e6449b6c9c7c6c425235edfd90b90da2f51c3f9a531508dd8cdefcac4b', 'src/dlbench/a2/engine.py': '8f9a6d00b2cfe02eaa648204e452c23a92a597573680373b130645c3bbdf0f71', 'src/dlbench/a2/losses.py': '85698eb02f893a643e2fd89370ce1177c2f055dc93b19604965c0f97a6d23fe4', 'src/dlbench/a2/metrics.py': '04e947369fffc1b26196aff09700456c48d026b200e0676b3c9a133b00bed43d', 'src/dlbench/a2/models/__init__.py': 'f96c8426b6966284279f668d552cfc5e1c6c76eca7624ea8d6c17061a6c262f6', 'src/dlbench/a2/models/segformer.py': '571cf469302c78292a3c39094dd961512b0ea0ac7685e51b18eceb504b35b780', 'src/dlbench/a2/models/unet.py': 'cfece5b7156edd2f8809b78f1c249500b80c70f964f15ec621bcd9d82c346f60', 'src/dlbench/a2/trainer.py': 'b7e2cfc6cc131ab25f7cd4e4a001f71c6a2f46d65aa77af91429c33f1738f326', 'configs/a2/protocol.py': 'f02307a516a4be2b7d80e3a3fa870c54a17cd62a7267e4157a50410e76658d21', 'configs/a2/segformer_off.py': '124f3b4ad13c505cdd874797ad4f0b842b63b1f89f533aa7ed76aaf17d7fb506', 'configs/a2/segformer_on.py': '42ed2b17377d0caf77f4d1a270ecb188a098e9c831227a5a8ea4caba5736f2a7', 'configs/a2/unet.py': 'a93ec8ff69ffbdcda49c30dc25de1e8627c7881354e417867f38b05471314d4f', 'src/dlbench/common/__init__.py': 'e6ac429d49908586b6d4b6f9caf370d1ac526f18c1df888f1aad32f8bc418f24', 'src/dlbench/common/artifacts.py': '9abe7a0e1b40a81fc6ef1c72e0bf588858cfb4f4e332632fc732154be974356c', 'src/dlbench/common/config.py': '284bc2109101c445e683de1ab1a43e64759d79ce461a58a5ba3e231af7a76f2a', 'src/dlbench/common/reproducibility.py': '342f3b921dc297c8ce347c47a8721a258be7cedf8c44e181ef366cd958e6b5a0', 'src/dlbench/__init__.py': '66b8937cffe23de942ef7de82516911a50753c5765ca56a0394300aaef9a0676', 'helpers/a2_support.py': 'd628fde2c2bee63a37b3c50e71681b8501974f685ba7f56d6135d2004385a4a4', 'helpers/a2_resume.py': 'a98e3a0f8ec4bdb6dab1d867203ee142997fe6ca6e6ba5458aed345f3b216644', 'helpers/a2_test.py': 'fdbf23dd6686ffbe6e9fdfc51a637f4b70fcd68c041aae5f033b2f6ae4ddc482', 'helpers/kaggle_runner.py': '4da4d491e22cda6fe673536dafe3ec8414262c57c5e9d6817dbf152defb7379a', 'manifest.json': '5ee6745c85aebfbd0e5bc51c8b7cca62f40b4c5dbcacaf6dfe27fa52fdc712c8', 'helpers/a2_reference.py': 'd23afc1bc4853633224aba035c401bb93e1c87a55093c89ca47cf1dba53629f1', 'completed_seed69420.json': '1365acf6b5ec066c3c5e3120ee39328e70d2aabb9d7172d047970a23da2a6c64'}
BUNDLE_SHA256 = 'ed2a3ed646e39c2790c45314d732d021a16f82ca9195173c38820010d0c78c0b'
BUILD_ROOT = Path('/kaggle/temp/dlbench_a2_bundle_' + BUNDLE_SHA256[:12])
BUILD_ROOT.mkdir(parents=True, exist_ok=True)
embedded = json.loads(zlib.decompress(base64.b64decode(PAYLOAD)))
assert set(embedded) == set(EXPECTED_SHA256)
for relative, encoded_file in embedded.items():
    rel = Path(relative)
    assert not rel.is_absolute() and '..' not in rel.parts
    data = base64.b64decode(encoded_file)
    assert hashlib.sha256(data).hexdigest() == EXPECTED_SHA256[relative], relative
    target = BUILD_ROOT / rel
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        assert target.read_bytes() == data, 'Source tạm đã thay đổi: ' + relative
    else:
        target.write_bytes(data)
print('Verified embedded source/manifest/helpers:', len(embedded), 'files')


In [ ]:
# v2: dependency setup không gọi ensurepip, kể cả khi có venv tạo dở.
import subprocess
def prepare_environment(directory, packages):
    directory = Path(directory)
    subprocess.check_call([sys.executable, '-m', 'venv', '--without-pip',
                           '--system-site-packages', str(directory)])
    python = directory / ('Scripts/python.exe' if os.name == 'nt' else 'bin/python')
    manager = [sys.executable, '-m', 'pip']
    probe = subprocess.run(manager + ['--help'], capture_output=True, text=True)
    if probe.returncode != 0 or '--python' not in probe.stdout:
        import urllib.request
        import zipfile
        pip_zip = directory.parent / 'dlbench_pip_bootstrap_v2.pyz'
        if not pip_zip.is_file() or not zipfile.is_zipfile(pip_zip):
            print('Using the official PyPA pip zipapp; enable Kaggle Internet.')
            urllib.request.urlretrieve('https://bootstrap.pypa.io/pip/pip.pyz', str(pip_zip))
        manager = [sys.executable, str(pip_zip)]
    subprocess.check_call(manager + ['--python', str(python), 'install',
        '--disable-pip-version-check', '-q'] + list(packages))
    subprocess.check_call([str(python), '-c', 'import sys; print("Worker Python:", sys.executable)'])
    return python

VENV = Path('/kaggle/temp/dlbench_a2_venv_v2')
PYTHON = prepare_environment(VENV, [
    'Pillow==12.3.0', 'transformers==4.57.6', 'huggingface-hub==0.36.2',
    'gdown==5.2.0', 'tqdm==4.67.3', 'safetensors>=0.4.3'])
settings = {
    'output_root': str(OUTPUT_ROOT), 'input_root': '/kaggle/input',
    'repo': str(BUILD_ROOT), 'manifest': str(BUILD_ROOT / 'manifest.json'),
    'helpers': [str(BUILD_ROOT / 'helpers' / (n + '.py')) for n in
                ('a2_support', 'a2_resume', 'a2_test', 'kaggle_runner', 'a2_reference')],
    'completed_reference': str(BUILD_ROOT / 'completed_seed69420.json'),
    'bundle_sha256': BUNDLE_SHA256, 'cache': '/kaggle/temp/dlbench_a2_cache',
    'session_hours': float(MAX_SESSION_HOURS), 'session_started_unix': SESSION_STARTED_UNIX,
    'restore_from': RESTORE_FROM, 'data_archive': DATA_ARCHIVE,
    'run_main': RUN_MAIN, 'run_extras': RUN_EXTRAS, 'selected_runs': SELECT_RUNS,
}
SETTINGS = BUILD_ROOT / 'settings.json'
SETTINGS.write_text(json.dumps(settings, indent=2), encoding='utf-8')
driver = "import sys\nfrom pathlib import Path\nroot=Path(sys.argv[1])\nsys.path[:0]=[str(root/'helpers'),str(root/'src')]\nimport kaggle_runner\nkaggle_runner.main(sys.argv[2])\n"
DRIVER = BUILD_ROOT / 'run_suite.py'
DRIVER.write_text(driver, encoding='utf-8')
print('Plan: 2 remaining seeds × 3 configurations = 6 NEW runs; completed69420 reused. Output:', OUTPUT_ROOT)


## 3. Data → pilot → train/resume → test → gallery/timing → ZIP

- Tự tìm archive `dataset.zip` đã gắn ở Input; nếu chưa có, tải nguồn Drive đã dùng cho U-Net. Kiểm tra **553.029.970 byte** và SHA-256 EDA trước giải nén. Nếu Drive chặn/quyền truy cập không public, gắn archive gốc làm private Kaggle input rồi chạy lại. Không dùng `google.colab` hoặc mount Drive.
- Kiểm tra đủ train/validation ảnh và native-mask hashes; giữ split 5.057/1.077/1.078 và video-disjoint. Chạy GPU pilot 20 bước cho mỗi cấu hình ở67/69, pin cả6 config trước main/test.
- Checkpoint sau mỗi epoch; giữ metadata/source/history/optimizer/RNG. Resume chỉ tự xử lý RAM/kernel/processor drift qua continuation có provenance. Đổi GPU/library/precision/config sẽ dừng để tránh trộn thí nghiệm.
- Test chỉ sau completion/early stopping. Cache test có checkpoint/membership/output hashes. Run hoàn tất không train thêm epoch. Ảnh gallery là sáu test indices cố định, không phải tỷ lệ lỗi toàn bộ tập.
- Timing là forward-only FP32 batch4, 10 warm-up/50 lần đo; ms/image là thời gian batch chia4, **không phải latency batch1**. CUDA bilinear backward dùng deterministic warn-only, không bảo đảm GPU giống từng bit.

Khi dừng vì ngân sách, đây là trạng thái **paused**, không phải đã chạy xong cả6 run mới. Output phải được Kaggle lưu hoặc tải về; có file trong `/kaggle/working` chưa tự bảo đảm tồn tại sau khi session bị mất.


In [ ]:
# Cell chạy chính — có thể kéo dài nhiều giờ; checkpoint được lưu mỗi epoch.
import os, subprocess
env = dict(os.environ)
env.update({'CUBLAS_WORKSPACE_CONFIG': ':4096:8', 'USE_TF': '0', 'USE_FLAX': '0',
    'HF_HOME': '/kaggle/temp/dlbench_a2_hf', 'HF_HUB_DISABLE_XET': '1',
    'TOKENIZERS_PARALLELISM': 'false', 'TQDM_MININTERVAL': '15',
    'OMP_NUM_THREADS': '2', 'MKL_NUM_THREADS': '2', 'PYTHONNOUSERSITE': '1'})
OUTPUT_ROOT.parent.mkdir(parents=True, exist_ok=True)
LOG = OUTPUT_ROOT.parent / (SUITE_TAG + '_session.log')
with LOG.open('a', encoding='utf-8') as log:
    process = subprocess.Popen([str(PYTHON), '-u', str(DRIVER), str(BUILD_ROOT), str(SETTINGS)],
        env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True); log.write(line); log.flush()
        returncode = process.wait()
    except KeyboardInterrupt:
        # SIGINT lets the worker retain its last saved checkpoint and export in finally.
        process.send_signal(__import__('signal').SIGINT)
        raise
if returncode:
    raise RuntimeError(f'Worker dừng với mã {returncode}. Xem log/output diagnostic; giữ nguyên checkpoint. Log: {LOG}')


## Kết quả hai seed mới và tổng hợp ba seed

`per_run.csv`, `mean_std.csv`, `paired_deltas.csv` chỉ chứa67/69 chạy trên Kaggle; sample SD chỉ khi có≥2 seed thực tế.
`combined_three_seed_results.csv`, `combined_three_seed_mean_std.csv`, `combined_three_seed_paired_deltas.csv` thêm69420 đã xác minh.
Mỗi dòng ghi runtime profile và ngày/bằng chứng. Chỉ đánh dấu đủ3 seed khi thực sự có U/S0/S1 ở cả69420/67/69.

Mean/sample SD kết hợp mô tả các run có cả khác biệt seed và môi trường Colab/Kaggle; không khẳng định ổn định chỉ do seed hay tương đương runtime.
S0/S1 trong cùng seed vẫn phải dùng cùng runtime/init; timing các profile giữ riêng. Không dùng test để chọn seed/cấu hình.

Checkpoint, best/last, optimizer/RNG/history, validation/test cache, gallery/timing, ZIP và resume của v2 được giữ.
Sau khi paused, lưu output của v3 và gắn lại vào Input cho phiên tiếp theo. Nhiều output cùng tag cần chọn bản mới nhất rõ ràng.


In [ ]:
from IPython.display import display, FileLink
summary_path = OUTPUT_ROOT / 'summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text())
    print('New Kaggle test runs:', summary['completed_test_runs'], '/ 6')
    print('ALL SIX NEW RUNS COMPLETE:', summary['all_planned_test_complete'])
    combined = summary['combined_with_completed_seed69420']
    print('Total including completed69420:', combined['completed_total_test_runs'], '/ 9')
    print('ALL THREE SEEDS COMPLETE:', combined['all_three_seeds_test_complete'])
    print(combined['runtime_caveat'])
    print(json.dumps(summary['aggregates'], indent=2))
    for file in ('summary.json', 'per_run.csv', 'mean_std.csv', 'paired_deltas.csv', 'session_status.json', 'combined_three_seed_summary.json', 'combined_three_seed_results.csv', 'combined_three_seed_mean_std.csv', 'combined_three_seed_paired_deltas.csv'):
        path = OUTPUT_ROOT / file
        if path.is_file(): display(FileLink(str(path)))
package = OUTPUT_ROOT.parent / (SUITE_TAG + '_results.zip')
if package.is_file(): display(FileLink(str(package)))
print('Tải/lưu từ tab Output của Kaggle. Khi paused: gắn output mới nhất vào Input của phiên kế tiếp.')


Nguồn nền tảng: [Kaggle Notebooks documentation](https://www.kaggle.com/docs/notebooks).
Pretrained: [NVIDIA SegFormer-B0 pinned revision](https://huggingface.co/nvidia/segformer-b0-finetuned-ade-512-512/tree/489d5cd81a0b59fab9b7ea758d3548ebe99677da).
Dataset: [TrashCan](https://irvlab.cs.umn.edu/resources/trashcan). Đoạn code/data không thay thế license/attribution của nguồn.
Notebook được kiểm tra cấu trúc/cú pháp, payload và các fixture CPU; chạy thật trên Kaggle GPU vẫn do người dùng thực hiện.

Seed69420 retained: U-Net verified28/10/2026; S0/S1 verified03/11/2026. Chỉ tái sử dụng saved quality metrics/provenance, không chạy lại model69420.
